# 07 Neo4j: the same diagnosis, asked to the database

**Goal (Phase 1 of the research plan):** move the graph from Python (networkx) into **Neo4j**, and show that nothing changes.
Notebook 05 walks the graph in Python. Here the same walks are written as **Cypher queries** and sent to your Neo4j database.
The scoring rules are copied unchanged from notebook 05, step 2.

**Pass test:** for all 99 bad jobs, the `kg_rules` answer (cause and evidence ids) from Neo4j must equal the one from notebook 05.

**Before you run:**
1. Notebook 04 and notebook 05 have been run (this notebook compares with `results_kg_rules.csv`).
2. Your `trulaser` database is **Running** in Neo4j Desktop and `neo4j/import_kg.cypher` (in this repo) has been run on it.
3. Once, in the Anaconda Prompt: `pip install neo4j`

## 0. Setup: connect to the database

In [ ]:
from pathlib import Path
import pandas as pd
from neo4j import GraphDatabase

OUT = Path("..") / "data" / "processed"
URI = "bolt://localhost:7687"
PASSWORD = "trulaser123"            # the password you set when you created the instance

driver = GraphDatabase.driver(URI, auth=("neo4j", PASSWORD))
driver.verify_connectivity()

def q(cypher, **params):
    """run one Cypher query and return its rows as a list of dicts"""
    records, _, _ = driver.execute_query(cypher, params)
    return [r.data() for r in records]

key = pd.read_csv(OUT / "answer_key_jobs.csv")
print("connected |", len(key), "bad jobs in the answer key")

## 1. Is the database the same graph as notebook 04?
Count nodes and edges in Neo4j and compare with the two CSV files notebook 04 saved. If they differ, re-run `import_kg.cypher`.

In [ ]:
n_db = q("MATCH (n) RETURN count(n) AS n")[0]["n"]
e_db = q("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"]
n_csv, e_csv = len(pd.read_csv(OUT / "kg_nodes.csv")), len(pd.read_csv(OUT / "kg_edges.csv"))
print(f"nodes: Neo4j {n_db} | notebook 04 {n_csv}")
print(f"edges: Neo4j {e_db} | notebook 04 {e_csv}")
assert (n_db, e_db) == (n_csv, e_csv), "Neo4j holds a different graph: copy the new CSVs to the import folder and re-run import_kg.cypher"

## 2. What is "normal"? (alarm rates, now from Neo4j)
Same calculation as notebook 05 step 1: alarms per code divided by the number of days in the log.

In [ ]:
rows = q("MATCH (a:AlarmEvent) RETURN a.name AS code, a.time AS time")
alarm_times = pd.DataFrame([(r["code"], r["time"].to_native()) for r in rows], columns=["code", "time"])
n_days = (alarm_times["time"].max() - alarm_times["time"].min()).days + 1
NORMAL_PER_DAY = (alarm_times["code"].value_counts() / n_days).round(2).to_dict()
NORMAL_PER_DAY

## 3. Retrieve: the graph walks as Cypher
Each walk from notebook 05 becomes one short query. Read them as sentences:

| query | in words |
|---|---|
| `JOB` | this job and the defect the inspector found |
| `CANDIDATES` | the causes the guide lists for that defect, with rank and gas |
| `INDICATES` | which alarm codes point to which cause |
| `LAST_ALARMS` | the latest alarm of each code before this job |
| `LAST_REPAIRS` | the latest repair of each cause before this job |

The scoring below the queries is copied from notebook 05, so any difference in the result can only come from the database.

In [ ]:
JOB = """
MATCH (j:Job {name: $job})-[:HAS_DEFECT]->(d:Defect)
RETURN j {.*} AS job, d.name AS defect"""
CANDIDATES = """
MATCH (c:Cause)-[r:CAUSES]->(:Defect {name: $defect})
RETURN c.name AS cause, r.rank AS rank, r.gas AS gas"""
INDICATES = """
MATCH (k:AlarmCode)-[:INDICATES]->(c:Cause)
RETURN c.name AS cause, k.name AS code, k.text AS text"""
LAST_ALARMS = """
MATCH (:Job {name: $job})-[r:LAST_ALARM]->(a:AlarmEvent)
RETURN r.code AS code, r.hours_before AS hours_before, r.count_24h AS count_24h, a.id AS node"""
LAST_REPAIRS = """
MATCH (:Job {name: $job})-[r:AFTER_REPAIR]->(m:Maintenance)
RETURN r.cause AS cause, r.hours_since AS hours_since, m.id AS node, m.text AS text"""

# the knowledge layer is small and never changes, so read it once
codes_for = {}
code_text = {}
for r in q(INDICATES):
    codes_for.setdefault(r["cause"], []).append(r["code"])
    code_text[r["code"]] = r["text"]
code_text.update({r["code"]: r["text"] for r in q("MATCH (k:AlarmCode) RETURN k.name AS code, k.text AS text")})
fix_of = {r["cause"]: r["fix"] for r in q("MATCH (c:Cause) RETURN c.name AS cause, c.fix AS fix")}

In [ ]:
def retrieve_neo4j(job_id):
    row = q(JOB, job=job_id)[0]
    job, defect = row["job"], row["defect"]
    # CSV import stores these as text: turn them back into numbers / yes-no
    job["bevel_angle"] = float(job["bevel_angle"])
    job["is_fast"] = job["is_fast"] == "True"
    jn = f"job:{job_id}"

    rows = q(CANDIDATES, defect=defect)
    cands = [(r["rank"], r["cause"]) for r in rows if r["gas"] in ("any", job["gas"])]
    if not cands:                                   # e.g. dross on O2 with no O2 row: use all dross causes
        cands = [(r["rank"], r["cause"]) for r in rows]
    cands = [c for _, c in sorted(set(cands))]

    last_alarm = {r["code"]: (r["node"], r) for r in q(LAST_ALARMS, job=job_id)}
    last_repair = {r["cause"]: (r["node"], r) for r in q(LAST_REPAIRS, job=job_id)}
    repair_text = {r[0]: r[1]["text"] for r in last_repair.values()}
    evidence, scores, n = {}, {}, 0

    def add(cause, text, points):
        nonlocal n
        n += 1
        evidence.setdefault(cause, []).append(f"E{n}: {text}")
        scores[cause] = scores.get(cause, 0) + points

    # ---- from here on: the same rules as notebook 05, step 2 ----
    for cause in cands:
        scores[cause] = 0
        for code_ in codes_for.get(cause, []):
            if code_ in last_alarm and last_alarm[code_][1]["hours_before"] <= 24:
                a_node, e = last_alarm[code_]
                normal = NORMAL_PER_DAY.get(code_, 0)
                unusual = e["count_24h"] > max(1.5 * normal, 0.5)
                fixed = cause in last_repair and last_repair[cause][1]["hours_since"] < e["hours_before"]
                verdict = "repaired since then -> no longer counts" if fixed else "UNUSUAL" if unusual else "normal"
                add(cause, f"alarm {code_} ({code_text[code_]}) {e['count_24h']}x in the 24 h before the job, "
                           f"last one {e['hours_before']} h before start; normal is {normal}/day -> {verdict} [{a_node}]",
                    3 if unusual and not fixed else 0)
        if cause == "bevel_out_of_calibration" and "3301" in last_alarm:
            a_node, e = last_alarm["3301"]
            rep = last_repair.get(cause)
            not_fixed = rep is None or rep[1]["hours_since"] > e["hours_before"]
            add(cause, f"collision alarm 3301 {e['hours_before']} h before the job; "
                       f"{'NO recalibration since then' if not_fixed else 'recalibrated after it'}; bevel angle of this job {job['bevel_angle']} deg [{a_node}]",
                4 if not_fixed and job["bevel_angle"] != 0 else 0)
        if cause == "gas_supply_low" and cause in last_repair and "1201" in last_alarm:
            rep_node, r = last_repair[cause]
            if r["hours_since"] < last_alarm["1201"][1]["hours_before"]:
                add(cause, f"gas regulator replaced {r['hours_since']} h before the job, after the last 1201 alarm -> supply already fixed [{rep_node}]", -3)
        if cause == "glass_dirty":
            if cause in last_repair:
                rep_node, r = last_repair[cause]
                overdue = r["hours_since"] > 24
                add(cause, f"last glass cleaning {r['hours_since']} h before the job ('{repair_text[rep_node]}'); "
                           f"guide: check every shift -> {'OVERDUE' if overdue else 'ok'} [{rep_node}]", 2 if overdue else 0)
        if cause == "nozzle_worn":
            h = job["nozzle_hours"]
            add(cause, f"nozzle operating hours {h} h; guide: replace every 40-60 h -> {'WORN (above 60 h)' if h > 60 else 'ok'} [{jn}]",
                3 if h > 100 else 2 if h > 60 else 0)
        if cause == "feed_too_high":
            too_high = job["is_fast"] or job["feed_vs_table"] > 1.10
            add(cause, f"program {job['program']}, feed {job['feed_vs_table']}x the cutting table; guide: max 1.10x, _FAST programs not released "
                       f"-> {'TOO HIGH' if too_high else 'ok'} [{jn}]", 4 if too_high else 0)
        if cause not in evidence:
            evidence[cause] = ["no data in the logs to check this cause"]

    return {"job_id": job_id, "defect": defect, "job": job, "candidates": cands, "evidence": evidence, "scores": scores}

def diagnose_rules(r):
    best = max(r["candidates"], key=lambda c: (r["scores"][c], -r["candidates"].index(c)))
    used = [line.split(":")[0] for line in r["evidence"][best] if line.startswith("E")]
    return {"cause": best, "evidence_ids": used, "fix": fix_of[best]}

r = retrieve_neo4j("JOB-00391")
print(r["job_id"], "| defect:", r["defect"], "| candidates:", r["candidates"])
for c in r["candidates"]:
    print(f"\n{c}  (score {r['scores'][c]})")
    for line in r["evidence"][c]:
        print("   ", line)
diagnose_rules(r)

## 4. The pass test: all 99 jobs, Neo4j vs notebook 05
Run `kg_rules` on every bad job through Neo4j and compare with `results_kg_rules.csv` from notebook 05.
If every cause and every evidence id matches, the move to Neo4j is complete: the database answers exactly like the Python graph.

In [ ]:
import time
t0 = time.time()
rows = []
for job_id in key["job_id"]:
    ans = diagnose_rules(retrieve_neo4j(job_id))
    rows.append({"job_id": job_id, "cause": ans["cause"], "evidence_ids": " ".join(ans["evidence_ids"])})
neo = pd.DataFrame(rows)
neo.to_csv(OUT / "results_neo4j_rules.csv", index=False)
print(f"{len(neo)} jobs in {time.time() - t0:.1f} s")

nx_ = pd.read_csv(OUT / "results_kg_rules.csv").fillna({"evidence_ids": ""})
both = neo.merge(nx_[["job_id", "cause", "evidence_ids"]], on="job_id", suffixes=("_neo4j", "_networkx"))
same = (both["cause_neo4j"] == both["cause_networkx"]) & (both["evidence_ids_neo4j"] == both["evidence_ids_networkx"])
print(f"same answer and evidence: {same.sum()} of {len(both)} jobs")
assert same.all(), both[~same]

acc = (neo.merge(key, on="job_id", suffixes=("", "_key")).eval("cause == cause_key")).mean()
print(f"strict accuracy through Neo4j: {acc:.1%}")

## What this notebook showed
- The knowledge graph now lives in a **database** (Neo4j), not only in a Python file.
- Every walk notebook 05 does in Python is a short **Cypher** query here, readable enough to put in the report.
- **Pass test:** same cause and same evidence on all 99 jobs, so the Phase 1 move changes nothing in the results.

**For the report:** one table with the five queries (step 3), the pass-test line, and a Neo4j Browser screenshot of one job.

**Next:** photos (Phase 2). They become new nodes attached to the `Inspection` node of their job.